#### 03 - Dependency Parsing Evaluation

In this notebook, we will create the initial framework for evaluating our dependency parsing system against a gold standard using the eval.py tool provided by ud-tools.

#### Getting system outputs

In [1]:
# Set up paths
SYS_TREEBANK_PATH = "../data/dep_parsing_eval/system_output.conllu" # where we direct the system output
GOLD_TREEBANK_PATH = "../data/dep_parsing_eval/gold_standard.conllu" # should already contain manually annotated trees
GOLD_SENTENCES_PATH = "../data/dep_parsing_eval/gold_sentences.txt" # text file with 1 sentence per line (same order as gold treebank)


In [2]:
# FST and CG3 setup
from fst_runtime.fst import Fst
import sys 
sys.path.append("../src")
from cg3_process import disambiguate
from dependency_parsing import cg3_to_conllu_batch

fst_binary_filename = "../data/fst/ojibwe.att"
fst = Fst(fst_binary_filename)
cg3_grammar_file = "../data/CG3_rules/Ojibwe_updated_dep_parsing.cg3"

In [4]:
# Get sentences in gold standard
sentences_to_parse = []
with open(GOLD_SENTENCES_PATH, 'r', encoding="utf-8") as file:
    sentences_to_parse = file.readlines()

# Parse sentences and direct output to SYS_TREEBANK_PATH
for sentence in sentences_to_parse:
    disambiguated_sentence = disambiguate(sentence, cg3_grammar_file, fst)
    print(disambiguated_sentence)
    cg3_to_conllu_batch(disambiguated_sentence, SYS_TREEBANK_PATH) 

"<mina'>"
	"mina'" VTA Imp Sim 2SgSubj 3SgProxObj ID:1
"<a'awe>"
	"a'awe" PRONDem NA ProxSg ID:2 R:Dep_Obj_V:1
"<noondayaabaagwed>"
"<.>"


✓ appended sentence #1 to system_output.conllu
"<namanj>"
	"namanj" ADVDub
"<gaa-izhinamogwen>"
	"izhinam" PVSub/gaa VAI Cnj Pos Dub 3SgProxSubj ID:2
	"izhinam" PVTense/gii VAI Pcp Pos Dub 3SgProxSubj 3SgProxHead ID:2
"<iwidi>"
	"iwidi" ADVLoc
"<agwajiing>"
	"agwajiing" ADVLoc
"<wa'awe>"
	"wa'awe" PRONDem NA ProxSg ID:5 R:Dep_Subj_V:2 R:Dep_Subj_V:6
"<gaa-piindigeyaanimizid>"
	"biindigeyaanimizi" PVTense/gii VAI Pcp Pos Neu 3SgProxSubj 3SgProxHead ID:6
"<.>"


✓ appended sentence #2 to system_output.conllu
"<ogii-nookwezwaan>"
	"nookwezwi" PVTense/gii VTA Ind Pos Neu 3SgProxSubj 3SgObvObj ID:1
	"nookwez" PVTense/gii VTA Ind Pos Neu 3SgProxSubj 3SgObvObj ID:1
"<a'aw>"
	"a'aw" PRONDem NA ProxSg ID:2 R:Dep_Dem_N:3
"<mindimooyenh>"
	"mindimooyenh" NA ProxSg ID:3 R:Dep_Subj_V:1
"<ini>"
	"ini" PRONDem NA ObvSg ID:4 R:Dep_Obj_V:1 R:Dep_Subj_V:6
"<gaa-...>

#### Evaluation using eval.py

In [5]:
import subprocess, sys, re, pandas as pd, urllib.request, io, zipfile, shutil
from pathlib import Path

# 1) Reference gold, system output files and eval tool
gold_path   = Path(GOLD_TREEBANK_PATH)  
system_path = Path(SYS_TREEBANK_PATH)  
assert gold_path.exists()
eval_py = Path("../ud-tools/eval.py")

# make sure files exist and are non-empty
for p, label in [(gold_path, "Gold"), (system_path, "System")]:
    if not p.is_file():
        raise FileNotFoundError(f"{label} file not found: {p}")
    if p.stat().st_size == 0:
        raise ValueError(f"{label} file is empty: {p}")

if not eval_py.is_file():           
    print("UD scorer not found")

# 2) run the scorer and capture stdout
cmd = [sys.executable, str(eval_py), "-v", str(gold_path), str(system_path)]
print("Running:", " ".join(cmd))

try:
    proc = subprocess.run(cmd, capture_output=True, text=True, check=True)
    out = proc.stdout
except subprocess.CalledProcessError as e:
    print("Eval script failed:")
    print("STDOUT:", e.stdout)
    print("STDERR:", e.stderr)
    raise

# 3) print output
print(out)

Running: /opt/miniconda3/envs/myenv_py312/bin/python ../ud-tools/eval.py -v ../data/dep_parsing_eval/gold_standard.conllu ../data/dep_parsing_eval/system_output.conllu
Metric     | Precision |    Recall |  F1 Score | AligndAcc
-----------+-----------+-----------+-----------+-----------
Tokens     |    100.00 |    100.00 |    100.00 |
Sentences  |    100.00 |    100.00 |    100.00 |
Words      |    100.00 |    100.00 |    100.00 |
UPOS       |     93.94 |     93.94 |     93.94 |     93.94
XPOS       |    100.00 |    100.00 |    100.00 |    100.00
UFeats     |    100.00 |    100.00 |    100.00 |    100.00
AllTags    |     93.94 |     93.94 |     93.94 |     93.94
Lemmas     |    100.00 |    100.00 |    100.00 |    100.00
UAS        |    100.00 |    100.00 |    100.00 |    100.00
LAS        |    100.00 |    100.00 |    100.00 |    100.00
CLAS       |    100.00 |    100.00 |    100.00 |    100.00
MLAS       |     92.31 |     92.31 |     92.31 |     92.31
BLEX       |    100.00 |    100.00 